# Module 2: Comparative Analysis of Keras and PyTorch Models

> **Assignment (10 points):** Compare the trained Keras and PyTorch CNN classifiers with the same evaluation helper.

In [ ]:
import os, sys, pathlib, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
PROJECT_ROOT = pathlib.Path(os.path.abspath(os.getcwd()))
for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (p / 'images_dataSAT').is_dir():
        PROJECT_ROOT = p
        break
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

## Shared evaluation helper: `print_metrics`

In [ ]:
def print_metrics(y_true, y_pred, model_name):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    cm   = confusion_matrix(y_true, y_pred)
    print(f'\n========== {model_name} ==========')
    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall   : {rec:.4f}')
    print(f'F1 score : {f1:.4f}')
    print(f'Confusion matrix:\n{cm}')

    auc = roc_auc_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True)
    os.makedirs('models', exist_ok=True)
    plt.savefig(f'models/roc_{model_name.replace(" ", "_")}.png')
    plt.show()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'confusion_matrix': cm, 'auc': auc}

### Task 1 (Question): What does the code `preds > 0.5` in `preds = (preds > 0.5).astype(int).flatten()` do?

**Answer:** The model's output is a sigmoid probability in the range [0, 1]. `preds > 0.5` compares every probability with the 0.5 threshold and returns a boolean array (True → predicted positive class, False → negative class). `.astype(int)` converts the booleans to integer labels 1/0 and `.flatten()` turns the array into a single one-dimensional array. The whole line therefore converts continuous probability scores into hard binary predictions.

## Keras model: reload, predict, evaluate

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

keras_model = tf.keras.models.load_model('models/cnn_model.keras')

datagen = ImageDataGenerator(rescale=1. / 255)
test_generator = datagen.flow_from_directory(
    './images_dataSAT', target_size=(64, 64), batch_size=32,
    class_mode='binary', shuffle=False, seed=42)

y_true_keras = test_generator.classes
y_prob_keras = keras_model.predict(test_generator, verbose=0).flatten()
y_pred_keras = (y_prob_keras > 0.5).astype(int).flatten()

### Task 2: Print the performance metrics for the Keras model using `print_metrics`

In [ ]:
keras_metrics = print_metrics(y_true_keras, y_pred_keras, 'Keras CNN Model')

### Task 3 (Question): What is the significance of the F1 score?

**Answer:** The F1 score is the **harmonic mean of precision and recall** (F1 = 2·P·R / (P + R)). It provides a single balanced measure of model performance and is especially useful for **imbalanced datasets**: a high F1 means the model is good both at not missing positives (high recall) and at not producing false positives (high precision).

## PyTorch model: reload, predict, evaluate

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

pt_model = CNNClassifier()
pt_model.load_state_dict(torch.load('models/pytorch_cnn.pth', map_location='cpu', weights_only=True))
pt_model.eval()

val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_dataset = datasets.ImageFolder('./images_dataSAT', transform=val_transform)
val_dataset, _ = random_split(val_dataset, [1800, 4200])
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

y_true_torch = []
y_prob_torch = []
with torch.no_grad():
    for images, labels in val_loader:
        out = torch.sigmoid(pt_model(images))
        y_prob_torch.extend(out.squeeze(1).tolist())
        y_true_torch.extend(labels.tolist())

y_true_torch = np.array(y_true_torch)
y_prob_torch = np.array(y_prob_torch)
y_pred_torch = (y_prob_torch > 0.5).astype(int).flatten()

### Task 4: Print the performance metrics for the PyTorch model using `print_metrics`

In [ ]:
torch_metrics = print_metrics(y_true_torch, y_pred_torch, 'PyTorch CNN Model')

### Task 5 (Question): What is the total number of false negatives in the confusion matrix for the PyTorch model evaluated above?

In [ ]:
cm = torch_metrics['confusion_matrix']
fn = int(cm[1][0])          # class 1 (agricultural) predicted as class 0 (non-agricultural)
print('Confusion matrix:')
print(cm)
print()
print(f'Total number of false negatives (FN) for the PyTorch model: {fn}')

**Answer:** A false negative means an agricultural image (class 1) was wrongly predicted as non-agricultural (class 0). Looking at the confusion matrix of the PyTorch model, the bottom-left cell (row = class 1, column = class 0) gives the total FN count, printed in the cell above.

## Side-by-side summary

In [ ]:
print(f'{"Metric":<12}{"Keras CNN":<14}{"PyTorch CNN":<14}')
for key in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    print(f'{key:<12}{keras_metrics[key]:<14.4f}{torch_metrics[key]:<14.4f}')